In [ ]:
import pandas as pd
import numpy as np
import random

# =========================================================
# ASSUMPTIONS
# =========================================================
# Existing DataFrames:
#
df_suppliers = pd.read_csv("data/raw/df_suppliers.csv")
df_po=pd.read_csv("data/raw/df_purchase_orders.csv")
df_shipments=pd.read_csv("data/raw/df_shipments.csv")
df_quality_inspections=pd.read_csv("data/raw/df_quality_inspections.csv")
#
# Required columns:
#
# PO:
# po_id
# supplier_id
# component_id
# received_quantity
#
# QUALITY:
# shipment_id
# po_id
# rejected_units
#
# =========================================================


# =========================================================
# PLANT MAPPING
# =========================================================

plant_map = {

    "electronics": [
        "PLANT_IND_001",
        "PLANT_IND_002"
    ],

    "metals": [
        "PLANT_US_001",
        "PLANT_IND_004"
    ],

    "packaging": [
        "PLANT_IND_003"
    ],

    "chemicals": [
        "PLANT_IND_005"
    ],

    "plastics": [
        "PLANT_IND_006"
    ],

    "textiles": [
        "PLANT_IND_007"
    ]
}


# =========================================================
# MERGE QUALITY DATA
# =========================================================

quality_summary = (

    df_quality_inspections

    .groupby("po_id")["rejected_units"]

    .sum()

    .reset_index()

)


quality_summary.rename(

    columns={
        "rejected_units":
        "total_rejected_units"
    },

    inplace=True
)


# =========================================================
# MERGE INTO PO
# =========================================================

inventory_base = df_po.merge(

    quality_summary,

    on="po_id",

    how="left"

)


inventory_base[
    "total_rejected_units"
] = inventory_base[
    "total_rejected_units"
].fillna(0)


# =========================================================
# USABLE INVENTORY
# =========================================================

inventory_base["usable_inventory"] = (

    inventory_base["received_quantity"]
    - inventory_base["total_rejected_units"]

)

inventory_base["usable_inventory"] = (

    inventory_base["usable_inventory"]
    .clip(lower=0)

)


# =========================================================
# BUILD INVENTORY TABLE
# =========================================================

inventory_rows = []

inventory_counter = 1


for _, row in inventory_base.iterrows():

    supplier = df_suppliers[

        df_suppliers["supplier_id"]
        == row["supplier_id"]

    ].iloc[0]


    category = supplier["supplier_category"]

    criticality = supplier["criticality_level"]

    country = supplier["country"]


    # =====================================================
    # PLANT ASSIGNMENT
    # =====================================================

    plant_id = random.choice(

        plant_map.get(
            category,
            ["PLANT_MISC_001"]
        )
    )


    # =====================================================
    # DAILY CONSUMPTION
    # =====================================================
    # Consumption varies by category
    # =====================================================

    consumption_ranges = {

        "electronics": (20, 150),

        "metals": (50, 300),

        "packaging": (200, 1000),

        "chemicals": (30, 120),

        "plastics": (40, 200),

        "textiles": (25, 100)

    }


    low, high = consumption_ranges.get(
        category,
        (10, 50)
    )


    avg_daily_consumption = round(

        np.random.uniform(low, high),

        2
    )


    # =====================================================
    # SIMULATED CONSUMPTION PERIOD
    # =====================================================

    consumption_days = np.random.randint(
        5,
        90
    )


    # ---------------------------------------------------------
    # CONSUMPTION SHOULD NOT EXCEED
    # MOST OF AVAILABLE INVENTORY
    # ---------------------------------------------------------

    max_consumption_ratio = np.random.uniform(
        0.20,
        0.85
    )

    consumed_inventory = (

        row["usable_inventory"]

        * max_consumption_ratio
    )


    # =====================================================
    # CURRENT STOCK
    # =====================================================

    current_stock = (

        row["usable_inventory"]
        - consumed_inventory

    )


    # No negative inventory
    current_stock = max(
        round(current_stock, 2),
        0
    )


    # =====================================================
    # SAFETY STOCK
    # =====================================================

    safety_multiplier = 1.0


    # International suppliers need more buffer
    if country != "India":

        safety_multiplier += 0.50


    # Critical suppliers require more protection
    if criticality >= 4:

        safety_multiplier += 0.50


    lead_time_days = np.random.randint(5, 20)

    safety_stock = (

        avg_daily_consumption

        * lead_time_days

        * np.random.uniform(0.8, 1.5)

    )


    # =====================================================
    # CREATE RECORD
    # =====================================================

    inventory_rows.append({

        "inventory_id":
            f"INV{inventory_counter:07}",

        "component_id":
            row["component_id"],

        "supplier_id":
            row["supplier_id"],

        "current_stock":
            current_stock,

        "avg_daily_consumption":
            avg_daily_consumption,

        "safety_stock":
            safety_stock,

        "plant_id":
            plant_id
    })


    inventory_counter += 1


# =========================================================
# FINAL INVENTORY DATAFRAME
# =========================================================

df_inventory_status = pd.DataFrame(
    inventory_rows
)


# =========================================================
# OPTIONAL VALIDATIONS
# =========================================================

print("\n==============================")
print("INVENTORY SUMMARY")
print("==============================")

print(
    f"Total Inventory Records: "
    f"{len(df_inventory_status)}"
)

print(
    "\nLow Stock Situations:"
)

low_stock = df_inventory_status[

    df_inventory_status["current_stock"]
    <
    df_inventory_status["safety_stock"]

]

print(len(low_stock))


print("\nPlant Distribution:")

print(
    df_inventory_status[
        "plant_id"
    ].value_counts()
)

print("\nSample Records:\n")

print(
    df_inventory_status.head()
)

In [ ]:
df_inventory_status.to_csv("data/raw/df_inventory.csv", index=False)